# Retail Bronze ETL — AWS Glue / PySpark

This notebook documents the Bronze-layer ingestion logic from the exported AWS Glue job `retail-bronze-etl`.

## Job configuration
- **Glue version:** 6.0
- **Worker type:** G.1X
- **Workers:** 2
- **Timeout:** 15 minutes
- **Bookmark:** job-bookmark-disable

> The code below preserves the script embedded in the Glue export. To execute it, use a Spark environment with access to the specified S3 paths.


## ETL code


In [ ]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    current_timestamp,
    input_file_name,
    regexp_extract
)

# --------------------------------------------------


## Initialize Spark session


In [ ]:
# 1. Initialize Spark session
# --------------------------------------------------

spark = SparkSession.builder.appName("Retail-Bronze-ETL").getOrCreate() 

# --------------------------------------------------


## Define S3 paths


In [ ]:
# 2. Define S3 paths
# --------------------------------------------------

SOURCE_BUCKET = "s3://tus-retail-clothing-store-datasets"

BRONZE_PATH = f"{SOURCE_BUCKET}/bronze"

datasets = {
    "brands" : f"{SOURCE_BUCKET}/brands",
    "category": f"{SOURCE_BUCKET}/category/",
    "customers": f"{SOURCE_BUCKET}/customers/",
    "date": f"{SOURCE_BUCKET}/date/",
    "products": f"{SOURCE_BUCKET}/products/",
    "order_items": f"{SOURCE_BUCKET}/landing/"
}


# --------------------------------------------------


## Function to process each dataset


In [ ]:
# 3. Function to process each dataset
# --------------------------------------------------

def process_dataset(dataset_name, source_path):
    print(f"Starting Bronze processing: {dataset_name}")
    # Read CSV while preserving source values as strings
    df = spark.read \
        .format("csv") \
        .option("header", "true") \
        .option("inferSchema", "false") \
        .option("mode", "PERMISSIVE") \
        .option("quote", '"') \
        .option("escape", '"') \
        .load(source_path)
        
    # Add ingestion metadata
    df = df.withColumn("ingestion_timestamp", current_timestamp())
    
    df = df.withColumn(
        "source_file",
        regexp_extract(input_file_name(), r"([^/]+)$", 1)
    )
    
    # Define destination
    target_path = f"{BRONZE_PATH}/{dataset_name}/"

    # Write as Parquet
    df.write \
        .mode("overwrite") \
        .format("parquet") \
        .save(target_path)

    print(f"Completed: {dataset_name}")
    print(f"Destination: {target_path}")

# --------------------------------------------------


## Process all six datasets


In [ ]:
# 4. Process all six datasets
# --------------------------------------------------

for dataset_name, source_path in datasets.items():
    process_dataset(dataset_name, source_path)
    
# --------------------------------------------------


## Finish


In [ ]:
# 5. Finish
# --------------------------------------------------

print("All six datasets processed successfully!")
